# Chapter 3: Fluid Characterization and PVT Modeling

Real reservoir fluids contain heavy hydrocarbon fractions that cannot be represented by
individual pure components. This chapter demonstrates:

- **Plus-fraction characterization**: Splitting C7+ into pseudo-components using TBP data
- **Phase envelope**: Mapping the two-phase boundary of a gas condensate
- **Liquid dropout (CVD)**: How liquid volume changes with pressure depletion
- **GOR and Bo**: Key PVT parameters for production forecasting

We model a gas condensate fluid typical of deep-water or HP/HT reservoirs.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


In [2]:
import matplotlib.pyplot as plt
import numpy as np

from neqsim import jneqsim

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

## 3.1 Creating a Gas Condensate Fluid with Plus Fractions

The fluid includes light gases (N₂, CO₂, C1–C5) and heavier TBP fractions
(C7 through C10+) characterized by molar mass and specific gravity.

In [3]:
def create_gas_condensate(T_K, P_bara):
    """Create a gas condensate fluid with TBP plus fractions."""
    fluid = SystemSrkEos(T_K, P_bara)
    fluid.addComponent("nitrogen", 0.34)
    fluid.addComponent("CO2", 3.59)
    fluid.addComponent("methane", 67.42)
    fluid.addComponent("ethane", 7.46)
    fluid.addComponent("propane", 3.05)
    fluid.addComponent("i-butane", 0.85)
    fluid.addComponent("n-butane", 1.68)
    fluid.addComponent("i-pentane", 0.63)
    fluid.addComponent("n-pentane", 0.74)
    fluid.addComponent("n-hexane", 1.10)
    fluid.addTBPfraction("C7", 1.0, 92.0 / 1000.0, 0.7324)
    fluid.addTBPfraction("C8", 1.15, 104.0 / 1000.0, 0.7602)
    fluid.addTBPfraction("C9", 0.95, 119.0 / 1000.0, 0.7824)
    fluid.addTBPfraction("C10", 5.0, 230.0 / 1000.0, 0.85)
    fluid.setMixingRule("classic")
    fluid.setMultiPhaseCheck(True)
    return fluid

# Test flash
fluid_test = create_gas_condensate(273.15 + 90.0, 200.0)
ops_test = ThermodynamicOperations(fluid_test)
ops_test.TPflash()
fluid_test.initProperties()
print(f"Number of phases: {fluid_test.getNumberOfPhases()}")
print(f"Total density: {fluid_test.getDensity('kg/m3'):.2f} kg/m³")
print(f"Number of components: {fluid_test.getNumberOfComponents()}")

Number of phases: 2
Total density: 278.04 kg/m³
Number of components: 14


## 3.2 Figure 1 — Molecular Weight Distribution

After characterization, the plus fractions are split into pseudo-components.
This bar chart shows the molecular weight of each component in the fluid.

In [4]:
fluid_mw = create_gas_condensate(273.15 + 90.0, 200.0)
ops_mw = ThermodynamicOperations(fluid_mw)
ops_mw.TPflash()
fluid_mw.initProperties()

n_comp = fluid_mw.getNumberOfComponents()
comp_names = []
mol_weights = []
mole_fracs = []

for i in range(n_comp):
    name = fluid_mw.getComponent(i).getComponentName()
    mw = fluid_mw.getComponent(i).getMolarMass() * 1000.0  # kg/mol -> g/mol
    zf = fluid_mw.getComponent(i).getz()
    comp_names.append(name)
    mol_weights.append(mw)
    mole_fracs.append(zf)

fig, ax = plt.subplots(figsize=(12, 5))
x_pos = np.arange(len(comp_names))
colors = ['#1f77b4' if mw < 80 else '#ff7f0e' if mw < 150 else '#d62728' for mw in mol_weights]
ax.bar(x_pos, mol_weights, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(comp_names, rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Component', fontsize=12)
ax.set_ylabel('Molecular Weight (g/mol)', fontsize=12)
ax.set_title('Figure 3.1: Molecular Weight Distribution of Gas Condensate Components', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../figures/fig01_molecular_weight_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_9816\4121916217.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.3 Figure 2 — Phase Envelope (PT Diagram)

The phase envelope shows the boundary between single-phase and two-phase regions.
For a gas condensate, the cricondentherm and cricondenbar define the maximum
temperature and pressure at which two phases can coexist.

In [5]:
fluid_env = create_gas_condensate(273.15 + 90.0, 200.0)
ops_env = ThermodynamicOperations(fluid_env)

try:
    ops_env.calcPTphaseEnvelope(True, 1.0)
except Exception:
    # Retry without arguments
    try:
        ops_env.calcPTphaseEnvelope()
    except Exception as e2:
        print(f"Phase envelope calculation failed: {e2}")

dewT, dewP, bubT, bubP = [], [], [], []

try:
    dewT_raw = ops_env.get("dewT")
    dewP_raw = ops_env.get("dewP")
    if dewT_raw is not None and dewT_raw.length > 0:
        dewT = [dewT_raw[i] - 273.15 for i in range(dewT_raw.length)]
        dewP = [dewP_raw[i] for i in range(dewP_raw.length)]
except Exception:
    pass

try:
    bubT_raw = ops_env.get("bubT")
    bubP_raw = ops_env.get("bubP")
    if bubT_raw is not None and bubT_raw.length > 0:
        bubT = [bubT_raw[i] - 273.15 for i in range(bubT_raw.length)]
        bubP = [bubP_raw[i] for i in range(bubP_raw.length)]
except Exception:
    pass

fig, ax = plt.subplots(figsize=(9, 6))
if dewT:
    ax.plot(dewT, dewP, 'b-', linewidth=2.5, label='Dew point curve')
if bubT:
    ax.plot(bubT, bubP, 'r-', linewidth=2.5, label='Bubble point curve')

if not dewT and not bubT:
    # Generate approximate phase envelope for illustration
    T_dew = np.linspace(-80, 200, 50)
    P_dew = 350 * np.exp(-((T_dew - 50) / 120) ** 2)
    ax.plot(T_dew, P_dew, 'b-', linewidth=2.5, label='Dew point curve (approx.)')
    T_bub = np.linspace(-80, 100, 30)
    P_bub = 300 * np.exp(-((T_bub - 0) / 80) ** 2)
    ax.plot(T_bub, P_bub, 'r-', linewidth=2.5, label='Bubble point curve (approx.)')
    print("Note: Using approximate curves (phase envelope calc returned no data)")

ax.set_xlabel('Temperature (\u00b0C)', fontsize=12)
ax.set_ylabel('Pressure (bara)', fontsize=12)
ax.set_title('Figure 3.2: Phase Envelope of Gas Condensate', fontsize=13)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('../figures/fig02_phase_envelope_condensate.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Dew curve: {len(dewT)} points")
print(f"Bubble curve: {len(bubT)} points")

Dew curve: 66 points
Bubble curve: 0 points


C:\Users\ESOL\AppData\Local\Temp\ipykernel_9816\3197787766.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.4 Figure 3 — Liquid Dropout Curve (CVD-like)

During pressure depletion (constant volume depletion), liquid condenses from
the gas phase below the dew point. We simulate this by flashing at decreasing
pressures at reservoir temperature and tracking the liquid phase fraction.

In [6]:
T_res_K = 273.15 + 90.0
pressures_cvd = np.linspace(400, 20, 40)
liquid_vol_pct = []

for P in pressures_cvd:
    fluid = create_gas_condensate(T_res_K, float(P))
    ops = ThermodynamicOperations(fluid)
    ops.TPflash()
    fluid.initProperties()

    n_phases = fluid.getNumberOfPhases()
    if n_phases > 1 and fluid.hasPhaseType("oil"):
        # Liquid volume fraction as percentage
        oil_vol = fluid.getPhase("oil").getVolume("m3")
        total_vol = fluid.getVolume("m3")
        liquid_vol_pct.append(100.0 * oil_vol / total_vol)
    else:
        liquid_vol_pct.append(0.0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pressures_cvd, liquid_vol_pct, 'g-o', markersize=3, linewidth=2)
ax.set_xlabel('Pressure (bara)', fontsize=12)
ax.set_ylabel('Liquid Volume (%)', fontsize=12)
ax.set_title('Figure 3.3: Liquid Dropout Curve at 90°C (CVD-like)', fontsize=13)
ax.grid(True, alpha=0.3)
ax.invert_xaxis()
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('../figures/fig03_liquid_dropout_curve.png', dpi=150, bbox_inches='tight')
plt.show()

max_dropout = max(liquid_vol_pct)
max_p_idx = liquid_vol_pct.index(max_dropout)
print(f"Maximum liquid dropout: {max_dropout:.2f}% at {pressures_cvd[max_p_idx]:.0f} bara")

Maximum liquid dropout: 67.95% at 361 bara


C:\Users\ESOL\AppData\Local\Temp\ipykernel_9816\130556833.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.5 Figure 4 — Gas-Oil Ratio vs Pressure

The GOR increases as pressure drops below the saturation pressure,
because more gas is liberated from the liquid phase.

In [7]:
pressures_gor = np.linspace(400, 30, 35)
gor_values = []

for P in pressures_gor:
    fluid = create_gas_condensate(T_res_K, float(P))
    ops = ThermodynamicOperations(fluid)
    ops.TPflash()
    fluid.initProperties()

    n_phases = fluid.getNumberOfPhases()
    if n_phases > 1 and fluid.hasPhaseType("oil") and fluid.hasPhaseType("gas"):
        gas_vol = fluid.getPhase("gas").getVolume("m3")
        oil_vol = fluid.getPhase("oil").getVolume("m3")
        if oil_vol > 1e-12:
            gor_values.append(gas_vol / oil_vol)
        else:
            gor_values.append(float('nan'))
    else:
        gor_values.append(float('nan'))

# Filter valid points
valid_idx = [i for i, g in enumerate(gor_values) if not np.isnan(g)]
p_valid = [pressures_gor[i] for i in valid_idx]
gor_valid = [gor_values[i] for i in valid_idx]

fig, ax = plt.subplots(figsize=(8, 5))
if len(p_valid) > 0:
    ax.plot(p_valid, gor_valid, 'r-s', markersize=4, linewidth=2)
ax.set_xlabel('Pressure (bara)', fontsize=12)
ax.set_ylabel('Gas-Oil Ratio (vol/vol at conditions)', fontsize=12)
ax.set_title('Figure 3.4: GOR vs Pressure at 90°C', fontsize=13)
ax.grid(True, alpha=0.3)
ax.invert_xaxis()
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('../figures/fig04_gor_vs_pressure.png', dpi=150, bbox_inches='tight')
plt.show()
if len(gor_valid) > 0:
    print(f"GOR range: {min(gor_valid):.1f} – {max(gor_valid):.1f} (vol/vol at conditions)")

GOR range: 0.2 – 34.0 (vol/vol at conditions)


C:\Users\ESOL\AppData\Local\Temp\ipykernel_9816\3119865152.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3.6 Figure 5 — Formation Volume Factor (Bo) vs Pressure

The oil formation volume factor Bo is the ratio of oil volume at reservoir conditions
to its volume at standard conditions. We approximate it using density ratios:

$$B_o = \frac{\rho_{\text{std}}}{\rho_{\text{res}}}$$

In [8]:
# Reference density at standard conditions (1.01325 bara, 15°C)
fluid_std = create_gas_condensate(273.15 + 15.0, 1.01325)
ops_std = ThermodynamicOperations(fluid_std)
ops_std.TPflash()
fluid_std.initProperties()

rho_std = None
if fluid_std.hasPhaseType("oil"):
    rho_std = fluid_std.getPhase("oil").getDensity("kg/m3")
    print(f"Oil density at std conditions: {rho_std:.1f} kg/m³")
else:
    rho_std = fluid_std.getDensity("kg/m3")
    print(f"Using total fluid density at std conditions: {rho_std:.1f} kg/m³")

Oil density at std conditions: 819.7 kg/m³


In [9]:
pressures_bo = np.linspace(50, 400, 30)
bo_values = []

for P in pressures_bo:
    fluid = create_gas_condensate(T_res_K, float(P))
    ops = ThermodynamicOperations(fluid)
    ops.TPflash()
    fluid.initProperties()

    if fluid.hasPhaseType("oil"):
        rho_res = fluid.getPhase("oil").getDensity("kg/m3")
    else:
        rho_res = fluid.getDensity("kg/m3")

    if rho_res > 1e-6 and rho_std is not None:
        bo_values.append(rho_std / rho_res)
    else:
        bo_values.append(float('nan'))

valid_bo = [(p, b) for p, b in zip(pressures_bo, bo_values) if not np.isnan(b)]

fig, ax = plt.subplots(figsize=(8, 5))
if len(valid_bo) > 0:
    p_plot = [v[0] for v in valid_bo]
    b_plot = [v[1] for v in valid_bo]
    ax.plot(p_plot, b_plot, 'darkorange', marker='D', markersize=3, linewidth=2)
ax.set_xlabel('Pressure (bara)', fontsize=12)
ax.set_ylabel('Formation Volume Factor Bo (rb/stb)', fontsize=12)
ax.set_title('Figure 3.5: Bo vs Pressure at 90°C', fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('../figures/fig05_bo_vs_pressure.png', dpi=150, bbox_inches='tight')
plt.show()

if len(b_plot) > 0:
    print(f"Bo range: {min(b_plot):.3f} – {max(b_plot):.3f}")

Bo range: 1.133 – 1.936


C:\Users\ESOL\AppData\Local\Temp\ipykernel_9816\3926838383.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This chapter demonstrated:

1. **Plus-fraction characterization**: TBP fractions (C7–C10+) can be added to NeqSim fluids with molar mass and density
2. **Molecular weight distribution**: Heavier pseudo-components have MW > 200 g/mol and dominate liquid formation
3. **Phase envelope**: Gas condensate fluids show a wide two-phase region with cricondentherm typically above 200°C
4. **Liquid dropout**: Below the dew point, condensate drops out — maximum dropout occurs at an intermediate pressure
5. **GOR and Bo**: These PVT parameters change significantly with pressure and are essential for reserves estimation and production forecasting

Next chapter: **Reservoir Engineering Fundamentals** — inflow performance, reservoir deliverability, and material balance.